In [6]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
DATA_DIR = '/content/drive/MyDrive/nba-xgboost/data'

raw = pd.read_csv(f"{DATA_DIR}/games_raw_all.csv", parse_dates=['GAME_DATE'])
print(raw.shape)
raw.head(3)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(23958, 28)


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,22016,1610612758,SAC,Sacramento Kings,21601228,2017-04-12,SAC @ LAC,L,240,95,...,0.714,7,30,37,27,2,1,10,18,-20.0
1,22016,1610612762,UTA,Utah Jazz,21601227,2017-04-12,UTA vs. SAS,W,241,101,...,0.650,7,29,36,24,10,5,16,16,4.0
2,22016,1610612739,CLE,Cleveland Cavaliers,21601218,2017-04-12,CLE vs. TOR,L,240,83,...,0.708,10,32,42,20,5,7,15,14,-15.0


In [7]:
FEATURES = ['PTS','FG_PCT','FG3_PCT','FT_PCT','FGM','FGA','FG3M','FG3A',
            'FTM','FTA','AST','TOV','OREB','DREB','REB','STL','BLK','PF','PLUS_MINUS']
KEEP = ['SEASON_ID','TEAM_ID','TEAM_ABBREVIATION','GAME_ID','GAME_DATE','MATCHUP','WL'] + FEATURES

df = raw[KEEP].copy()
print(df.shape, len(FEATURES), "features")

(23958, 26) 19 features


In [8]:
before = len(df)
df = df.drop_duplicates(subset=['GAME_ID','TEAM_ID'])
print("Removed", before - len(df), "duplicate rows")

Removed 0 duplicate rows


In [9]:
print(df['SEASON_ID'].astype(str).str[0].value_counts())
df = df[df['SEASON_ID'].astype(str).str.startswith('2')]
print(df.shape)

SEASON_ID
2    23958
Name: count, dtype: int64
(23958, 26)


In [10]:
print(df.isnull().sum()[df.isnull().sum() > 0])
print("Rows with missing WL:", df['WL'].isnull().sum())

FT_PCT    1
dtype: int64
Rows with missing WL: 0


In [11]:
df = df.dropna(subset=FEATURES + ['WL', 'MATCHUP'])
print(df.shape)

(23957, 26)


In [12]:
counts = df.groupby('GAME_ID')['TEAM_ID'].transform('count')
print("Rows in incomplete games:", (counts != 2).sum())
df = df[counts == 2]
print(df.shape)

Rows in incomplete games: 1
(23956, 26)


In [13]:
df['IS_HOME'] = df['MATCHUP'].str.contains('vs.', regex=False).astype(int)

home_per_game = df.groupby('GAME_ID')['IS_HOME'].sum()
print(home_per_game.value_counts())

IS_HOME
1    11968
0       10
Name: count, dtype: int64


In [14]:
bad = home_per_game[home_per_game != 1].index
df = df[~df['GAME_ID'].isin(bad)]
print(df.shape)

(23936, 27)


In [15]:
print(df['WL'].value_counts())
df['WIN'] = (df['WL'] == 'W').astype(int)

WL
L    11968
W    11968
Name: count, dtype: int64


In [16]:
df = df.sort_values(['TEAM_ID','GAME_DATE','GAME_ID']).reset_index(drop=True)
df[['TEAM_ABBREVIATION','GAME_DATE','MATCHUP','WIN']].head(8)


,TEAM_ABBREVIATION,GAME_DATE,MATCHUP,WIN
0,ATL,2016-10-27,ATL vs. WAS,1
1,ATL,2016-10-29,ATL @ PHI,1
2,ATL,2016-10-31,ATL vs. SAC,1
3,ATL,2016-11-02,ATL vs. LAL,0
4,ATL,2016-11-04,ATL @ WAS,0
5,ATL,2016-11-05,ATL vs. HOU,1
6,ATL,2016-11-08,ATL @ CLE,1
7,ATL,2016-11-09,ATL vs. CHI,1


In [17]:
print("Shape:", df.shape)
print("Teams:", df['TEAM_ID'].nunique(), "(expect 30)")
print("Games:", df['GAME_ID'].nunique())
print("Nulls:", df.isnull().sum().sum())
print("Date range:", df['GAME_DATE'].min().date(), "->", df['GAME_DATE'].max().date())

# Is each team's date order really increasing?
ok = df.groupby('TEAM_ID')['GAME_DATE'].apply(lambda s: s.is_monotonic_increasing).all()
print("Dates sorted within every team:", ok)

# Games per season
print(df.groupby('SEASON_ID')['GAME_ID'].nunique())

Shape: (23936, 28)
Teams: 30 (expect 30)
Games: 11968
Nulls: 0
Date range: 2016-10-25 -> 2026-04-12
Dates sorted within every team: True
SEASON_ID
22016    1230
22017    1230
22018    1230
22019    1059
22020    1080
22021    1230
22022    1230
22023    1229
22024    1225
22025    1225
Name: GAME_ID, dtype: int64


In [18]:
df.to_csv(f"{DATA_DIR}/games_clean.csv", index=False)
print("Saved", df.shape)

Saved (23936, 28)
